In [1]:
import pandas as pd
import numpy as np

# Load both datasets
df_reddit = pd.read_csv("reddit_sample_sentiment.csv")
df_stocks = pd.read_csv("stock_data.csv")

df_reddit['created_date'] = pd.to_datetime(df_reddit['created_date'])
df_stocks['Date'] = pd.to_datetime(df_stocks['Date'])

print("Reddit shape:", df_reddit.shape)
print("Stocks shape:", df_stocks.shape)
print("\nReddit columns:", df_reddit.columns.tolist())
print("Stock columns:", df_stocks.columns.tolist())

Reddit shape: (200000, 15)
Stocks shape: (755, 9)

Reddit columns: ['created_date', 'subreddit', 'subreddit_id', 'author', 'body', 'year', 'fog', 'num_letters', 'num_words', 'num_sentences', 'num_polysyllabic_words', 'avg_words_per_sentence', 'avg_syllables_per_word', 'sentiment_score', 'sentiment_label']
Stock columns: ['Date', 'AAPL', 'AMZN', 'GME', 'GOOG', 'MSFT', 'NFLX', 'SPY', 'TSLA']


In [4]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')


# Select numeric features from Reddit dataset
reddit_features = ['fog', 'num_letters', 'num_words', 'num_sentences',
                   'num_polysyllabic_words', 'avg_words_per_sentence',
                   'avg_syllables_per_word', 'sentiment_score']

X = df_reddit[reddit_features].copy()
y = df_reddit['sentiment_label']  # target variable

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features shape:", X.shape)
print("Target distribution:\n", y.value_counts())

Features shape: (200000, 8)
Target distribution:
 sentiment_label
positive    140248
negative     53762
neutral       5990
Name: count, dtype: int64


In [5]:
# TECHNIQUE 1: VARIANCE THRESHOLD
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.1)
X_var = selector.fit_transform(X_scaled)

# Get retained features
retained = [reddit_features[i] for i in range(len(reddit_features)) if selector.get_support()[i]]
removed = [reddit_features[i] for i in range(len(reddit_features)) if not selector.get_support()[i]]

print("=== Variance Threshold Results ===")
print(f"Original features: {len(reddit_features)}")
print(f"Retained features: {len(retained)}")
print(f"Removed features: {removed}")
print(f"\nRetained: {retained}")

# Plot variances
variances = X_scaled.var(axis=0)
plt.figure(figsize=(10, 5))
colors = ['#2ecc71' if s else '#e74c3c' for s in selector.get_support()]
plt.bar(reddit_features, variances, color=colors)
plt.axhline(y=0.1, color='black', linestyle='--', label='Threshold = 0.1')
plt.title('Variance Threshold – Feature Variances', fontsize=14, fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Variance')
plt.xticks(rotation=45, ha='right')
plt.legend()
plt.tight_layout()
plt.savefig('technique1_variance.png', dpi=150)
plt.show()
print("Saved!")

=== Variance Threshold Results ===
Original features: 8
Retained features: 8
Removed features: []

Retained: ['fog', 'num_letters', 'num_words', 'num_sentences', 'num_polysyllabic_words', 'avg_words_per_sentence', 'avg_syllables_per_word', 'sentiment_score']
Saved!


In [6]:
pca = PCA()
pca.fit(X_scaled)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

print("=== PCA Results ===")
for i, (exp, cum) in enumerate(zip(explained, cumulative)):
    print(f"PC{i+1}: {exp:.4f} ({exp*100:.1f}%) | Cumulative: {cum*100:.1f}%")

n_95 = np.argmax(cumulative >= 0.95) + 1
print(f"\nComponents needed for 95% variance: {n_95}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.bar(range(1, len(explained)+1), explained*100, color='#0D7377')
ax1.set_title('PCA – Explained Variance per Component', fontweight='bold')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance (%)')

ax2.plot(range(1, len(cumulative)+1), cumulative*100, marker='o', color='#1A2B4A')
ax2.axhline(y=95, color='red', linestyle='--', label='95% threshold')
ax2.set_title('PCA – Cumulative Explained Variance', fontweight='bold')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Variance (%)')
ax2.legend()

plt.tight_layout()
plt.savefig('technique2_pca.png', dpi=150)
plt.show()
print("Saved!")

=== PCA Results ===
PC1: 0.4596 (46.0%) | Cumulative: 46.0%
PC2: 0.2552 (25.5%) | Cumulative: 71.5%
PC3: 0.1338 (13.4%) | Cumulative: 84.9%
PC4: 0.1170 (11.7%) | Cumulative: 96.6%
PC5: 0.0191 (1.9%) | Cumulative: 98.5%
PC6: 0.0111 (1.1%) | Cumulative: 99.6%
PC7: 0.0035 (0.4%) | Cumulative: 99.9%
PC8: 0.0006 (0.1%) | Cumulative: 100.0%

Components needed for 95% variance: 4
Saved!


In [7]:
from sklearn.ensemble import RandomForestClassifier

X_sample = X_scaled[:50000]
y_sample = y[:50000]

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_sample, y_sample)

importances = rf.feature_importances_
importance_df = pd.DataFrame({
    'Feature': reddit_features,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("=== Random Forest Feature Importance ===")
print(importance_df.to_string(index=False))

plt.figure(figsize=(10, 5))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='#1A2B4A')
plt.title('Random Forest – Feature Importance', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('technique3_rf_importance.png', dpi=150)
plt.show()
print("Saved!")

=== Random Forest Feature Importance ===
               Feature  Importance
       sentiment_score    0.972501
avg_syllables_per_word    0.005741
           num_letters    0.005713
             num_words    0.004701
                   fog    0.003474
avg_words_per_sentence    0.003138
num_polysyllabic_words    0.002624
         num_sentences    0.002108
Saved!


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

top_features = importance_df['Feature'].head(5).tolist()
print("Top features used:", top_features)

X_top = df_reddit[top_features]
X_top_scaled = StandardScaler().fit_transform(X_top)

X_train, X_test, y_train, y_test = train_test_split(
    X_top_scaled, y, test_size=0.2, random_state=42, stratify=y)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print("\n=== Sentiment Model Results ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=['positive','neutral','negative'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['positive','neutral','negative'],
            yticklabels=['positive','neutral','negative'])
plt.title('Sentiment Model – Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('sentiment_model_cm.png', dpi=150)
plt.show()
print("Saved!")

Top features used: ['sentiment_score', 'avg_syllables_per_word', 'num_letters', 'num_words', 'fog']

=== Sentiment Model Results ===
Accuracy: 0.9994

Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00     10752
     neutral       1.00      0.98      0.99      1198
    positive       1.00      1.00      1.00     28050

    accuracy                           1.00     40000
   macro avg       1.00      0.99      1.00     40000
weighted avg       1.00      1.00      1.00     40000

Saved!


In [9]:
import os
print(os.getcwd())

C:\Users\jerry
